# 02 — Pydantic v2

## Objectifs pédagogiques

À la fin de ce notebook, vous saurez :

- définir un `BaseModel` avec des champs typés
- profiter de la **validation** et de la **coercition** automatique
- écrire des `@field_validator` et `@model_validator`
- sérialiser avec `model_dump` et `model_dump_json`
- charger depuis un dict / JSON avec `model_validate`

## Prérequis — ce que vous connaissez déjà

À ce stade de la formation intermédiaire, vous maîtrisez :

- `@dataclass` (notebook précédent)
- type hints modernes
- décorateurs (en usage — voir jour 3 pour les écrire)

Ce que nous n'avons **pas encore vu** (et que nous n'utiliserons donc pas dans ce notebook) :

- `pydantic-settings` pour la configuration (formation Avancé)
- intégration avec FastAPI

## Plan

1. Installation et motivation
2. Premier `BaseModel`
3. Validation et coercition
4. Défauts et champs optionnels
5. `Field(...)` : contraintes et métadonnées
6. `@field_validator` : validations custom
7. `@model_validator` : validation inter-champs
8. Sérialisation : `model_dump`, `model_dump_json`
9. Chargement : `model_validate`, `model_validate_json`
10. Dataclass vs Pydantic — tableau de décision
11. Synthèse
12. Exercices

---

## 1. Installation et motivation

Pydantic v2 est **la** bibliothèque de validation de données structurées en Python. Installation : `uv add pydantic` (ou `pip install pydantic`). Elle propose :

- des **modèles** déclaratifs (comme des dataclasses) ;
- une **validation stricte ou permissive** au choix ;
- une **sérialisation** contrôlée (JSON, dict) ;
- un moteur très rapide écrit en Rust (pydantic-core).

In [ ]:
import pydantic
print(pydantic.VERSION)


---

## 2. Premier `BaseModel`

Un modèle Pydantic se déclare en héritant de `BaseModel`. Les champs sont des annotations typées, exactement comme dans une dataclass.

In [ ]:
from pydantic import BaseModel


class Utilisateur(BaseModel):
    id: int
    nom: str
    email: str


In [ ]:
u = Utilisateur(id=1, nom='Alice', email='a@ex.fr')


In [ ]:
u


In [ ]:
u.nom


---

## 3. Validation et coercition

Pydantic **valide** et, par défaut, **coerce** les valeurs vers le type attendu. Exemple : passer `'42'` à un champ `int` produit `42`.

In [ ]:
Utilisateur(id='42', nom='Alice', email='a@ex.fr').id


### Erreur explicite si le type ne peut pas être coercé

In [ ]:
from pydantic import ValidationError

try:
    Utilisateur(id='pasunint', nom='Alice', email='a@ex.fr')
except ValidationError as exc:
    print(exc)


---

## 4. Défauts et champs optionnels

Comme en dataclass, les champs peuvent avoir une valeur par défaut ou être optionnels.

In [ ]:
class Salle(BaseModel):
    nom: str
    capacite: int = 10
    description: str | None = None


In [ ]:
Salle(nom='Mars')


In [ ]:
Salle(nom='Venus', capacite=6, description='Salle avec visio')


---

## 5. `Field(...)` : contraintes et métadonnées

Pour contraindre un champ (min/max, longueur, regex), on utilise `Field`.

In [ ]:
from pydantic import BaseModel, Field


class Produit(BaseModel):
    nom: str = Field(min_length=1, max_length=100)
    prix: float = Field(gt=0, description='Prix en euros, hors taxe')
    stock: int = Field(ge=0, default=0)


In [ ]:
Produit(nom='Pain', prix=1.2, stock=50)


In [ ]:
try:
    Produit(nom='', prix=-1)
except ValidationError as exc:
    print(exc)


---

## 6. `@field_validator` : validations custom

Pour une règle qui ne se réduit pas à un `Field`. L'argument `mode='after'` reçoit la valeur **après** coercition ; `mode='before'` reçoit la valeur brute.

In [ ]:
from pydantic import BaseModel, field_validator


class Utilisateur(BaseModel):
    email: str
    nom: str

    @field_validator('email')
    @classmethod
    def _email_valide(cls, v: str) -> str:
        if '@' not in v:
            raise ValueError('email sans @')
        return v.lower()

    @field_validator('nom')
    @classmethod
    def _nom_valide(cls, v: str) -> str:
        if not v.strip():
            raise ValueError('nom vide')
        return v.strip().title()


In [ ]:
Utilisateur(email='Alice@EX.FR', nom='  alice martin  ')


---

## 7. `@model_validator` : validation inter-champs

Quand la validation dépend de **plusieurs** champs (ex. `debut < fin`).

In [ ]:
from pydantic import BaseModel, model_validator
from typing import Self


class Intervalle(BaseModel):
    debut: int
    fin: int

    @model_validator(mode='after')
    def _ordre(self) -> Self:
        if self.debut >= self.fin:
            raise ValueError('debut >= fin')
        return self


In [ ]:
Intervalle(debut=9, fin=18)


In [ ]:
try:
    Intervalle(debut=20, fin=10)
except ValidationError as exc:
    print(exc)


---

## 8. Sérialisation : `model_dump`, `model_dump_json`

In [ ]:
u = Utilisateur(email='alice@ex.fr', nom='Alice')
u.model_dump()


In [ ]:
u.model_dump_json()


---

## 9. Chargement : `model_validate`, `model_validate_json`

In [ ]:
Utilisateur.model_validate({'email': 'bob@ex.fr', 'nom': 'Bob'})


In [ ]:
Utilisateur.model_validate_json('{"email": "eve@ex.fr", "nom": "Eve"}')


---

## 10. Dataclass vs Pydantic — tableau de décision

| Besoin | Préférer |
|---|---|
| Objet interne, pas de I/O | `@dataclass` |
| Valeur immuable hachable | `@dataclass(frozen=True)` |
| Entrée depuis JSON / HTTP / CSV | **Pydantic** |
| Validation métier complexe | **Pydantic** |
| Sérialisation contrôlée (JSON out) | **Pydantic** |
| Vitesse pure, pas de deps | `@dataclass(slots=True)` |

**Règle** : Pydantic au **bord** (API, DB, fichiers), dataclasses au **cœur** (logique métier interne).

---

## Synthèse

| Élément | Rôle |
|---|---|
| `BaseModel` | Classe de base |
| `Field(gt=, max_length=, ...)` | Contraintes simples |
| `@field_validator('champ')` | Validation d'un champ |
| `@model_validator(mode='after')` | Validation inter-champs |
| `model_dump()`, `model_dump_json()` | Sérialisation |
| `model_validate()`, `model_validate_json()` | Chargement |
| `ValidationError` | Exception |


### Règles à retenir

1. **Pydantic au bord, dataclass au cœur.**
2. **Toujours `classmethod` sur `@field_validator`**.
3. **`@model_validator(mode='after')` avec `Self`** pour les règles inter-champs.
4. **`model_dump(mode='json')`** pour un dict JSON-ready.
5. **Ne dupliquez pas la validation** : si Pydantic peut faire, Pydantic fait.

---

## Exercices

Les exercices sont gradués. Tous utilisent des fonctions typées (PEP 604).

### Exercice 1 — Premier modèle *(facile)*

Définir `Article` avec `id: int`, `titre: str`, `publie: bool = False`. Créer une instance et afficher `model_dump()`.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="02_Pydantic_v2", exercice=1)


<details>
<summary>📖 Voir la correction</summary>

```python
from pydantic import BaseModel

class Article(BaseModel):
    id: int
    titre: str
    publie: bool = False

a = Article(id=1, titre='Premier post')
print(a.model_dump())
```

</details>

### Exercice 2 — Contraintes avec `Field` *(moyen)*

Définir `Produit` avec `nom: str` (non vide, max 50), `prix: float` (> 0), `quantite: int` (>= 0, défaut 0). Tester avec une valeur valide et une invalide.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="02_Pydantic_v2", exercice=2)


<details>
<summary>📖 Voir la correction</summary>

```python
from pydantic import BaseModel, Field, ValidationError

class Produit(BaseModel):
    nom: str = Field(min_length=1, max_length=50)
    prix: float = Field(gt=0)
    quantite: int = Field(ge=0, default=0)

print(Produit(nom='Pain', prix=1.2))
try:
    Produit(nom='', prix=-3)
except ValidationError as exc:
    print(exc)
```

</details>

### Exercice 3 — Email normalisé *(moyen)*

Définir `Utilisateur(nom, email)`. Écrire un `@field_validator` qui :
- `nom` : strip + title-case
- `email` : strip + lower, refuse s'il manque `'@'`

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="02_Pydantic_v2", exercice=3)


<details>
<summary>📖 Voir la correction</summary>

```python
from pydantic import BaseModel, field_validator

class Utilisateur(BaseModel):
    nom: str
    email: str

    @field_validator('nom')
    @classmethod
    def _nom(cls, v: str) -> str:
        return v.strip().title()

    @field_validator('email')
    @classmethod
    def _email(cls, v: str) -> str:
        v = v.strip().lower()
        if '@' not in v:
            raise ValueError('email sans @')
        return v

u = Utilisateur(nom='  alice MARTIN ', email='  ALICE@EX.FR ')
print(u)
```

</details>

---

## Ressources externes

### Documentation officielle
- [Pydantic v2 docs](https://docs.pydantic.dev/latest/)
- [Migration Guide v1 → v2](https://docs.pydantic.dev/latest/migration/)

### Lectures complémentaires
- Samuel Colvin, talks PyCon sur pydantic-core et le moteur Rust.